In [ ]:
from dotenv import load_dotenv
import os

# Cargar variables de entorno (.env)
load_dotenv()

# Verificar claves API críticas
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY no encontrada en .env")
if not os.getenv("LANGSMITH_API_KEY"):
    raise ValueError("LANGSMITH_API_KEY no encontrada en .env")

In [ ]:
import json
from langsmith import evaluate, Client
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Importar el agente compilado 'agent' desde la arquitectura existente
# Asegúrate de estar ejecutando este notebook desde la raíz del backend o con el path correcto
import sys
sys.path.append("../../..") # Ajustar para que encuentre el módulo 'ai' si es necesario

try:
    from ai.agents.conversational_assistant.agent import agent
except ImportError:
    # Fallback si el path relativo no funciona directamente
    import sys
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
    from ai.agents.conversational_assistant.agent import agent

In [ ]:
def build_legal_judge_prompt(question, retrieved_context, assistant_answer):
    return f"""
Eres un magistrado experto en Derecho Penal Boliviano.

Evalúa la respuesta de un asistente jurídico virtual basado en RAG.

CRITERIOS DE EVALUACIÓN (0-5):
1. Exactitud jurídica: ¿La respuesta cita y aplica correctamente la normativa boliviana? (0=Error grave, 5=Perfecto)
2. Fidelidad al contexto (Groundedness): ¿La respuesta se basa SOLAMENTE en el contexto recuperado? (0=Alucinación total, 5=100% fiel)
3. Calidad del razonamiento: ¿La explicación es lógica y coherente? (0=Incoherente, 5=Excelente)
4. Claridad técnica: ¿Es comprensible para un abogado? (0=Confuso, 5=Claro y profesional)

REGLAS ESTRICTAS:
- Si el asistente inventa artículos o leyes que NO están en el contexto -> Fidelidad = 0.
- Si la interpretación legal es errónea -> Exactitud = 0.
- Si el contexto está vacío y el asistente responde con conocimiento general -> Fidelidad = 1 (punible por no usar contexto).

--- DATOS DE LA EJECUCIÓN ---

CONTEXTO RECUPERADO:
{retrieved_context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA DEL ASISTENTE:
{assistant_answer}

--- FORMATO DE SALIDA (JSON) ---
Devuelve SOLO un objeto JSON válido con esta estructura (sin markdown):
{{
  "scores": {{
    "legal_accuracy": int,
    "context_fidelity": int,
    "legal_reasoning": int,
    "clarity": int
  }},
  "justification": "Breve explicación del veredicto (máx 2 lineas)",
  "final_verdict": "PASS | FAIL | NEEDS_IMPROVEMENT"
}}
"""

In [ ]:
# 1. Definir la función objetivo (Target Function)
# Esta función toma los inputs del dataset y devuelve la salida del agente

async def predict_assistant(inputs: dict):
    question = inputs["question"] # Asumiendo que el dataset tiene columna 'question'
    
    # Invocar al agente
    # El agente espera inputs: {"question": str}
    # Y devuelve state con: "generation" (str) y "documents" (List[Document])
    try:
        state = await agent.ainvoke({"question": question})
        
        answer = state.get("generation", "")
        docs = state.get("documents", [])
        
        # Formatear contexto para el juez
        context_text = "\n\n".join([f"[Doc {i+1}] {d.page_content}" for i, d in enumerate(docs)])
        
        return {
            "answer": answer,
            "context": context_text
        }
    except Exception as e:
        return {
            "answer": f"Error executing agent: {str(e)}",
            "context": ""
        }

In [ ]:
# 2. Definir el Evaluador (LLM-as-a-Judge)
# Usaremos gpt-4o-mini como solicitaste

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def legal_evaluator(run, example):
    # Extraer datos
    assistant_answer = run.outputs.get("answer", "")
    retrieved_context = run.outputs.get("context", "")
    question = example.inputs.get("question", "")
    
    # (Opcional) Si tu dataset tiene 'reference_answer' o 'answer', puedes usarlo para comparar
    # reference = example.outputs.get("answer", "")
    
    # Construir prompt
    prompt_text = build_legal_judge_prompt(question, retrieved_context, assistant_answer)
    
    # Invocar juez
    try:
        response = judge_llm.invoke([HumanMessage(content=prompt_text)])
        content = response.content.strip()
        
        # Limpieza básica de markdown json
        if content.startswith("```json"):
            content = content[7:]
        if content.endswith("```"):
            content = content[:-3]
        
        result = json.loads(content)
        scores = result.get("scores", {})
        
        # Calcular puntaje promedio normalizado (0.0 a 1.0)
        # Suma de los 4 criterios (max 20) -> / 20
        total_score = (
            scores.get("legal_accuracy", 0) +
            scores.get("context_fidelity", 0) +
            scores.get("legal_reasoning", 0) +
            scores.get("clarity", 0)
        )
        avg_score = total_score / 20.0
        
        return {
            "key": "legal_quality_score",
            "score": avg_score,
            "comment": result.get("justification", "No justification provided")
        }
    except Exception as e:
        return {
            "key": "legal_quality_score",
            "score": 0.0,
            "comment": f"JUDGE ERROR: {str(e)}"
        }

In [ ]:
# 3. Ejecutar la Evaluación
# Dataset: "Knatuta AI Evaluation Dataset v6"

experiment_results = evaluate(
    predict_assistant,
    data="Knatuta AI Evaluation Dataset v6",
    evaluators=[legal_evaluator],
    experiment_prefix="gpt4o-mini-legal-judge",
    metadata={
        "version": "1.0.0",
        "judge_model": "gpt-4o-mini",
        "agent_model": "conversational_assistant_v1"
    }
)